# N-BaIoT — Step 6: Model Training (All Devices)

This notebook trains a Random Forest baseline independently for every device using the reduced datasets created in Step 5.

For each device:
1. Load its selected feature list from `feature_selection/selected_features.json`.
2. Create a bounded, class-balanced sample from `train_selected.csv`.
3. Fit a Random Forest classifier using training data only.
4. Evaluate on the complete `test_selected.csv` in chunks.
5. Save the trained model, class mapping, metrics, and confusion matrix.

**Leakage control:** Test data is used only for final evaluation. It is not used for sampling, fitting, tuning, or feature selection.

**Scope:** These are within-device experiments. A separate cross-device experiment is needed to test generalization to unseen devices.


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)

# ================= CONFIGURATION =================
DATASET_ROOT = Path.cwd()
PREPROCESSED_ROOT = DATASET_ROOT / "preprocessed"
CHUNK_SIZE = 100_000

# Training sample cap per class; limits RAM and training time.
MAX_TRAIN_SAMPLES_PER_CLASS = 20_000

N_ESTIMATORS = 300
MAX_DEPTH = 25
MIN_SAMPLES_LEAF = 2
MAX_FEATURES = "sqrt"
RANDOM_SEED = 42
N_JOBS = -1

print("Preprocessed folder:", PREPROCESSED_ROOT.resolve())
print("Max training sample/class:", MAX_TRAIN_SAMPLES_PER_CLASS)


## 6.1 Discover devices with Step 5 outputs

Only devices with selected training/testing files and a saved feature list are included.


In [ ]:
if not PREPROCESSED_ROOT.is_dir():
    raise FileNotFoundError(f"Cannot find {PREPROCESSED_ROOT}. Check DATASET_ROOT.")

device_dirs = sorted(
    p for p in PREPROCESSED_ROOT.iterdir()
    if p.is_dir()
    and (p / "feature_selection" / "train_selected.csv").is_file()
    and (p / "feature_selection" / "test_selected.csv").is_file()
    and (p / "feature_selection" / "selected_features.json").is_file()
)

print(f"Devices ready for training: {len(device_dirs)}")
for p in device_dirs:
    print("✓", p.name)

if not device_dirs:
    raise FileNotFoundError(
        "No Step 5 outputs found. Run Step5_Feature_Selection_All_Devices.ipynb first."
    )


## 6.2 Build a bounded training sample

Training files can be large. This cell uses random-priority reservoirs to retain at most `MAX_TRAIN_SAMPLES_PER_CLASS` training rows per class, while scanning the complete training file in chunks.

This is a practical baseline strategy for constrained RAM. The final test evaluation below still uses every test row.


In [ ]:
def make_training_sample(train_path, feature_cols):
    rng = np.random.default_rng(RANDOM_SEED)
    reservoirs = {}
    rows_seen = {}

    usecols = feature_cols + ["label", "label_id"]
    for chunk_no, chunk in enumerate(
        pd.read_csv(train_path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False),
        start=1
    ):
        ids = chunk["label_id"].to_numpy(dtype=np.int64)
        X = chunk[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        if not np.isfinite(X).all():
            raise ValueError(
                f"Invalid feature value in {train_path}, chunk {chunk_no}. "
                "Check preprocessing Step 2."
            )

        for class_id in np.unique(ids):
            class_id = int(class_id)
            idx = np.flatnonzero(ids == class_id)
            X_new = X[idx]
            y_new = np.full(len(idx), class_id, dtype=np.int64)
            priorities_new = rng.random(len(idx))
            rows_seen[class_id] = rows_seen.get(class_id, 0) + len(idx)

            if class_id in reservoirs:
                old_p, old_X, old_y = reservoirs[class_id]
                priorities = np.concatenate([old_p, priorities_new])
                X_all = np.concatenate([old_X, X_new], axis=0)
                y_all = np.concatenate([old_y, y_new])
            else:
                priorities, X_all, y_all = priorities_new, X_new, y_new

            keep_n = min(MAX_TRAIN_SAMPLES_PER_CLASS, len(priorities))
            keep = np.argpartition(priorities, keep_n - 1)[:keep_n]
            reservoirs[class_id] = (priorities[keep], X_all[keep], y_all[keep])

        if chunk_no % 10 == 0:
            print(f"  Scanned {chunk_no} training chunks")

    if not reservoirs:
        raise ValueError(f"No training samples found in {train_path}")

    X_parts, y_parts, sample_report = [], [], {}
    for class_id in sorted(reservoirs):
        _, X_class, y_class = reservoirs[class_id]
        X_parts.append(X_class)
        y_parts.append(y_class)
        sample_report[str(class_id)] = {
            "rows_available_in_train": int(rows_seen[class_id]),
            "rows_used_for_fit": int(len(y_class))
        }

    X_sample = np.vstack(X_parts)
    y_sample = np.concatenate(y_parts)
    order = rng.permutation(len(y_sample))
    return X_sample[order], y_sample[order], sample_report


## 6.3 Evaluate the complete test set in chunks

The model is evaluated against every row in `test_selected.csv`. We accumulate the true and predicted labels, then calculate accuracy, macro/weighted precision, recall and F1, plus a confusion matrix and per-class report.


In [ ]:
def evaluate_in_chunks(model, test_path, feature_cols, class_ids):
    y_true_parts = []
    y_pred_parts = []

    usecols = feature_cols + ["label_id"]
    for chunk_no, chunk in enumerate(
        pd.read_csv(test_path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False),
        start=1
    ):
        X = chunk[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=np.float32)
        if not np.isfinite(X).all():
            raise ValueError(
                f"Invalid feature value in {test_path}, chunk {chunk_no}. "
                "Check preprocessing Step 2."
            )
        y = chunk["label_id"].to_numpy(dtype=np.int64)
        pred = model.predict(X)
        y_true_parts.append(y)
        y_pred_parts.append(pred)

        if chunk_no % 10 == 0:
            print(f"  Evaluated {chunk_no} test chunks")

    if not y_true_parts:
        raise ValueError(f"No test rows found in {test_path}")

    y_true = np.concatenate(y_true_parts)
    y_pred = np.concatenate(y_pred_parts)

    labels = sorted(set(class_ids) | set(y_true.tolist()) | set(y_pred.tolist()))
    return y_true, y_pred, labels


## 6.4 Train and evaluate one model per device

A separate Random Forest is fitted for each device. The model uses the feature list selected in Step 5. Test data is not used during fitting.


In [ ]:
all_results = []

for device_dir in device_dirs:
    device_name = device_dir.name
    fs_dir = device_dir / "feature_selection"
    train_path = fs_dir / "train_selected.csv"
    test_path = fs_dir / "test_selected.csv"
    selected_path = fs_dir / "selected_features.json"

    model_dir = device_dir / "model"
    model_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 78)
    print("MODEL TRAINING:", device_name)
    print("=" * 78)

    with open(selected_path, "r", encoding="utf-8") as f:
        selection_info = json.load(f)

    feature_cols = selection_info["selected_features"]
    train_columns = set(pd.read_csv(train_path, nrows=0).columns)
    test_columns = set(pd.read_csv(test_path, nrows=0).columns)

    required = set(feature_cols) | {"label", "label_id"}
    if not required.issubset(train_columns):
        raise ValueError(f"{device_name}: selected train file is missing required columns.")
    if not required.issubset(test_columns):
        raise ValueError(f"{device_name}: selected test file is missing required columns.")

    print("Selected features:", len(feature_cols))
    print("Creating training sample...")
    X_train, y_train, sample_report = make_training_sample(train_path, feature_cols)

    train_classes = np.unique(y_train)
    print("Training sample shape:", X_train.shape)
    print("Training classes:", len(train_classes))

    if len(train_classes) < 2:
        raise ValueError(f"{device_name}: fewer than two classes in training sample.")

    model = RandomForestClassifier(
        n_estimators=N_ESTIMATORS,
        max_depth=MAX_DEPTH,
        min_samples_leaf=MIN_SAMPLES_LEAF,
        max_features=MAX_FEATURES,
        class_weight="balanced_subsample",
        random_state=RANDOM_SEED,
        n_jobs=N_JOBS
    )

    print("Fitting Random Forest...")
    model.fit(X_train, y_train)

    model_path = model_dir / "random_forest_model.joblib"
    joblib.dump(model, model_path)

    print("Evaluating complete test set...")
    y_true, y_pred, labels = evaluate_in_chunks(
        model, test_path, feature_cols, train_classes.tolist()
    )

    accuracy = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average="weighted", zero_division=0
    )

    report_dict = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    report_text = classification_report(
        y_true, y_pred, labels=labels, zero_division=0
    )
    matrix = confusion_matrix(y_true, y_pred, labels=labels)

    id_to_label = {}
    for value in sorted(set(y_true.tolist()) | set(y_pred.tolist())):
        matching = pd.read_csv(
            device_dir / "preprocessing_report.json"
        ) if False else None
        id_to_label[str(int(value))] = str(int(value))

    # Prefer the shared mapping file if it exists in the preprocessing root.
    mapping_candidates = [
        PREPROCESSED_ROOT / "label_mapping.json",
        DATASET_ROOT / "label_mapping.json"
    ]
    mapping = None
    for candidate in mapping_candidates:
        if candidate.is_file():
            with open(candidate, "r", encoding="utf-8") as f:
                mapping = json.load(f)
            break

    if mapping and "id_to_label" in mapping:
        id_to_label = {str(k): v for k, v in mapping["id_to_label"].items()}

    label_names = [id_to_label.get(str(int(v)), str(int(v))) for v in labels]
    cm_df = pd.DataFrame(matrix, index=label_names, columns=label_names)
    cm_df.to_csv(model_dir / "confusion_matrix.csv")

    with open(model_dir / "classification_report.txt", "w", encoding="utf-8") as f:
        f.write(report_text)

    metrics = {
        "device": device_name,
        "model": "RandomForestClassifier",
        "features_used": feature_cols,
        "training_sample_rows": int(len(y_train)),
        "training_sample_by_class": sample_report,
        "test_rows_evaluated": int(len(y_true)),
        "accuracy": float(accuracy),
        "macro_precision": float(p_macro),
        "macro_recall": float(r_macro),
        "macro_f1": float(f1_macro),
        "weighted_precision": float(p_weighted),
        "weighted_recall": float(r_weighted),
        "weighted_f1": float(f1_weighted),
        "classification_report": report_dict,
        "class_ids_in_confusion_matrix": [int(x) for x in labels],
        "class_names_in_confusion_matrix": label_names,
        "random_seed": RANDOM_SEED,
        "test_set_used_only_for_evaluation": True,
        "evaluation_scope": "Within-device stratified holdout; not cross-device generalization"
    }

    with open(model_dir / "evaluation_metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)

    result = {
        "device": device_name,
        "test_rows": len(y_true),
        "accuracy": accuracy,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "weighted_f1": f1_weighted
    }
    all_results.append(result)

    print("\nClassification report:")
    print(report_text)
    print(f"✓ Saved model and evaluation outputs to {model_dir}")


## 6.5 Compare results across devices

This summary compares the within-device test metrics. Because each device has its own model and its own holdout test set, do not interpret this as a cross-device generalization score.


In [ ]:
summary = pd.DataFrame(all_results)
display(summary)

summary_path = PREPROCESSED_ROOT / "all_devices_model_evaluation_summary.csv"
summary.to_csv(summary_path, index=False)

print("✓ Step 6 complete.")
print("Summary saved to:", summary_path)


## 6.6 Outputs and interpretation

For each device:

```text
preprocessed/
  Device_Name/
    model/
      random_forest_model.joblib
      evaluation_metrics.json
      classification_report.txt
      confusion_matrix.csv
```

Before reporting results:
- Confirm every expected device was processed.
- Check the test row count against Step 4.
- Review macro-F1 as well as accuracy; class imbalance can make accuracy misleading.
- Inspect the confusion matrix to see which attack classes are confused.
- Remember that the model was fitted on a bounded, class-balanced sample of the training set, not every training row.
- Do not tune hyperparameters against the final test set. Use a validation split from training data for tuning.

This is the first baseline model. Step 7 should compare alternative algorithms and/or tuned configurations using a validation strategy, then preserve a final untouched test evaluation.
